# 🎯 RF-DETR-nano Training — BGMI Killfeed Detection

**Ye notebook kya karta hai:** YouTube se collect kiye gaye BGMI frames pe RF-DETR-nano model train karta hai, jo game screen pe killfeed text detect karega.

## Kaise chalaye (sirf 5 steps)

1. **Kaggle pe account banao** (kaggle.com — free hai)
2. **Dataset upload karo:** Neeche Cell 2 me dataset ka naam dalo (guide docs me hai)
3. **GPU on karo:** Right side me ⚙️ Settings → Accelerator → **GPU T4** select karo
4. **Run dabao:** Top me ▶️ "Run All" dabao, phir chai piyo ☕
5. **Model download karo:** Last cell se trained model download karke AEGIS me lagao

⏱️ **Time:** 100k images pe ~6-10 ghante (T4 GPU). Beech me band mat karo.

---
## Hyperparameters (research-based)

| Setting | Value | Kyun |
|---|---|---|
| Learning rate | 1e-4 | RF-DETR official recommendation (AdamW ke saath) |
| Batch size | 4 × 4 grad-accum = 16 effective | T4 ke 16GB VRAM me fit hota hai |
| Epochs | 15 | COCO pretrained se fine-tune kar rahe hain, 100k images pe zyada nahi chahiye |
| Resolution | 672 | 56 se divisible (RF-DETR requirement), 640 se zyada = chhota text behtar dikhega |
| Optimizer | AdamW | Transformer models ke liye standard |

Sources: RF-DETR official docs (rfdetr.roboflow.com), ICLR 2026 paper, Roboflow benchmarks.

In [ ]:
# Cell 1: Packages install karo
!pip install -q "rfdetr[onnxexport]" supervision
print("✅ Packages installed")

In [ ]:
# Cell 2: Dataset load karo (Kaggle Datasets se)
# 👇 APNA DATASET PATH YAHAN DALO
DATASET_PATH = "/kaggle/input/bgmi-killfeed-dataset"  # <-- change karo agar naam alag hai

import os
assert os.path.exists(DATASET_PATH), f"Dataset nahi mila: {DATASET_PATH}"
assert os.path.exists(os.path.join(DATASET_PATH, "data.yaml")), "data.yaml nahi mila!"
print(f"✅ Dataset mil gaya: {DATASET_PATH}")

# Kitne images hain, gin lo
train_imgs = len(os.listdir(os.path.join(DATASET_PATH, "train", "images")))
val_imgs = len(os.listdir(os.path.join(DATASET_PATH, "valid", "images")))
print(f"📊 Train: {train_imgs:,} images | Val: {val_imgs:,} images")

In [ ]:
# Cell 3: GPU check karo
import torch
assert torch.cuda.is_available(), "❌ GPU nahi mila! Settings → Accelerator → GPU T4 karo"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4: TRAINING SHURU 🚀
# Ye cell sabse lamba chalega (6-10 ghante). Beech me interrupt mat karo.

from rfdetr import RFDETRNano

model = RFDETRNano(
    resolution=672,  # 56 se divisible, chhote text ke liye 640 se behtar
)

model.train(
    dataset_dir=DATASET_PATH,
    epochs=15,              # COCO pretrained se fine-tune — 100k images pe kaafi
    batch_size=4,           # T4 VRAM me fit
    grad_accum_steps=4,     # effective batch = 16 (official recommendation)
    lr=1e-4,                # official RF-DETR learning rate
    output_dir="/kaggle/working/rfdetr-killfeed",
)

In [ ]:
# Cell 5: Evaluation — model kitna achha hai? (imandaar numbers)

from rfdetr import RFDETRNano
import os

model = RFDETRNano(
    resolution=672,
    checkpoint="/kaggle/working/rfdetr-killfeed/checkpoint_best_total.pth",
)

# Validation set pe test karo
results = model.predict(
    os.path.join(DATASET_PATH, "valid", "images"),
)
print("✅ Evaluation complete — neeche metrics dekho")
print("📝 Note: Ye numbers validation set pe hain. Real game pe thode kam honge.")
# Detailed mAP ke liye: rfdetr ke built-in val tools use karo ya pycocotools

In [ ]:
# Cell 6: ONNX me export karo (AEGIS me lagane ke liye)

from rfdetr import RFDETRNano

model = RFDETRNano(
    resolution=672,
    checkpoint="/kaggle/working/rfdetr-killfeed/checkpoint_best_total.pth",
)

# ONNX export
model.export(output_dir="/kaggle/working/rfdetr-killfeed-onnx")
print("✅ ONNX model ready: /kaggle/working/rfdetr-killfeed-onnx/")
print("")
print("📥 AB KYA KARNA HAI:")
print("1. Right panel me 'Output' me jao")
print("2. rfdetr-killfeed-onnx folder download karo")
print("3. AEGIS me VISION_MODEL_PATH me ye path dalo")
print("4. Done! 🎉")